In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import dblquad

In [ ]:
# ------------------------------------------------------------------
# C_{X,X}(t,s)
# ------------------------------------------------------------------

def C_XX(t, s, alpha):


    u = sp.Min(s, t)
    v = sp.Max(s, t)

    return (
        (2*alpha*u**2*v)/((alpha-4)*(alpha-3))
        * (alpha-4 - (alpha-3)*v + v**(alpha-3))
        + (u/(alpha-2))
        * (alpha*u - 2*u**(alpha-1))
        * (
            1
            - (v/(alpha-2))
            * (alpha*v - 2*v**(alpha-1))
        )
    )

In [ ]:
# ------------------------------------------------------------------
# C_{Y,Y}(t,s)
# ------------------------------------------------------------------

def C_YY(t, s, alpha):


    u = sp.Min(s, t)
    v = sp.Max(s, t)

    return (
        (alpha*u - u**alpha)
        * (alpha - 1 - alpha*v + v**alpha)
        / (alpha - 1)**2
    )

In [ ]:
# ------------------------------------------------------------------
# C_{X,Y}(t,s)
# ------------------------------------------------------------------

def C_XY(t, s, alpha):


    # Cas s <= t
    expr_st = (
        alpha * (
            2*s**alpha/(alpha*(alpha-2))
            - s**2/(alpha-3)
            * (
                t
                - t**(alpha-2)/(alpha-2)
            )
        )
        + (s/(alpha-2))
        * (alpha*s - 2*s**(alpha-1))
        * (
            2
            - (alpha*t - t**alpha)/(alpha-1)
        )
    )

    # Cas t < s
    expr_ts = (
        alpha*t * (
            t**(alpha-1)/(alpha*(alpha-1))
            + s**2/(alpha-3)
            * (
                2*s**(alpha-3)/(alpha-1)
                - 1
            )
        )
        + s/((alpha-2)*(alpha-1))
        * (alpha*s - 2*s**(alpha-1))
        * (t**alpha - alpha*t)
        + (2*t)/((alpha-1)*(alpha-2))
        * (
            alpha*(alpha-1)*s
            - (
                alpha*s**(alpha-1)
                + (alpha-2)*t**(alpha-1)
            )
        )
    )

    return sp.Piecewise(
        (expr_st, s <= t),   # inclut le cas s=t
        (expr_ts, True)
    )

In [ ]:
# ------------------------------------------------------------------
# Densité f_Y(s)
# ------------------------------------------------------------------

def f_Y(s, alpha):


    return alpha/(alpha - 1) * (1 - s**(alpha - 1))


In [ ]:
# ------------------------------------------------------------------
# Densité f_T(t)
# ------------------------------------------------------------------

def f_T(t, alpha):


    return (2*alpha*(t - t**(alpha - 1))) / (alpha - 2)

In [ ]:
# ------------------------------------------------------------
# sigma(alpha) par intégration numérique
# ------------------------------------------------------------

def sigma_numeric(alpha):

    # sécurité sur alpha
    if alpha in [1, 2, 3, 4]:
        return np.nan


    # -------- IXX --------
    def integrand_xx(t, s):
        return C_XX(t, s, alpha) * f_Y(t, alpha) * f_Y(s, alpha)

    Ixx, _ = dblquad(
        integrand_xx,
        0, 1,          # s
        lambda s: 0,
        lambda s: 1    # t
    )

    # -------- IYY --------
    def integrand_yy(t, s):
        return C_YY(t, s, alpha) * f_T(t, alpha) * f_T(s, alpha)

    Iyy, _ = dblquad(
        integrand_yy,
        0, 1,
        lambda s: 0,
        lambda s: 1
    )

    # -------- IXY (découpage domaine) --------

    # t <= s
    def integrand_xy1(t, s):
        return C_XY(t, s, alpha) * f_T(t, alpha) * f_Y(s, alpha)

    Ixy1, _ = dblquad(
        integrand_xy1,
        0, 1,
        lambda s: 0,
        lambda s: s
    )

    # t > s
    def integrand_xy2(t, s):
        return C_XY(t, s, alpha) * f_T(t, alpha) * f_Y(s, alpha)

    Ixy2, _ = dblquad(
        integrand_xy2,
        0, 1,
        lambda s: s,
        lambda s: 1
    )

    Ixy = Ixy1 + Ixy2
#    print(Ixx)
#    print(Iyy)
#    print(Ixy)

    return 36 * (Ixx + Iyy - 2 * Ixy)

In [ ]:
alpha_vals = np.arange(0.5, 7., 0.07);
Nbmax=alpha_vals.shape[0];
var_theo=np.zeros(Nbmax);
i=0;
for aa in alpha_vals:
    print(aa,i)
    var_theo[i]=sigma_numeric(aa);
    print(var_theo[i])
    i=i+1;

In [ ]:
np.savetxt("alpha_vals_vartheo.dat",alpha_vals)
np.savetxt("var_theo_CVM.dat", var_theo)

In [ ]:
plt.plot(alpha_vals,var_theo)